# MCMC exploration

Visual walk-through of `bayes_cep.mcmc`: builds the `LogPosterior` for the single retained patient (`example_data/raw/`, `example_data/preprocessing/`), then samples it with `MCMCBuilder` -- the function-space MALA (pCNL) sampler, with pCN selectable via `MCMCSettings.algorithm`. Samples are written to a Zarr store.

The posterior is sampled relative to the prior as reference measure $\mu_0 = \mathcal N(\bar m, C)$, with the likelihood cost as potential $\Phi$ (see `bayes_cep.mcmc.measures`). The chain starts at the MAP estimate cached by `scripts/run_map.py` in `example_data/map/`.

This notebook is for interactive/visual inspection only.

In [25]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import zarr
from ls_bayesian.common.logging import BaseLogger, LoggerSettings
from ls_bayesian.mcmc.sampler import SamplerSettings

from bayes_cep.mcmc.builder import (
    MCMCAlgorithmName,
    MCMCBuilder,
    MCMCSettings,
    ZarrStorageSettings,
)
from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.posterior.builder import PosteriorBuilder, PosteriorSettings
from bayes_cep.posterior.eikonal_map import EikonalSolverSettings
from bayes_cep.posterior.likelihood import LikelihoodSettings
from bayes_cep.posterior.prior import PriorSettings
from bayes_cep.statistics.axial_statistics import (
    compute_axial_data_diff,
    compute_axial_mean_and_variance_blockwise,
)
from bayes_cep.statistics.correlation_length import (
    CorrelationLengthSettings,
    estimate_correlation_length,
)

pv.set_jupyter_backend("trame")

DATA_DIR = Path.cwd().parent / "example_data"
RAW_DIR = DATA_DIR / "raw"
PREPROCESSING_DIR = DATA_DIR / "preprocessing"
MAP_DIR = DATA_DIR / "map"
MCMC_DIR = DATA_DIR / "mcmc"

## Load preprocessed data

In [26]:
mesh = load_pyvista_mesh(RAW_DIR / "mesh.vtu")
basis_vectors = np.load(RAW_DIR / "basis_vecs.npy")
ground_truth_angle_field = np.load(PREPROCESSING_DIR / "ground_truth_angle_field.npy")
prior_mean_angle_field = np.load(PREPROCESSING_DIR / "prior_mean_angle_field.npy")
observed_vertex_indices = np.load(PREPROCESSING_DIR / "observed_vertex_indices.npy")
observed_activation_times = np.load(PREPROCESSING_DIR / "observed_activation_times.npy")
map_estimate = np.load(MAP_DIR / "map_estimate.npy")

interpolation_strategy = NearestNeighborInterpolationStrategy()
connectivity = mesh.cells.reshape(-1, 4)[:, 1:]
vertex_to_simplex_matrix = interpolation_strategy.assemble_matrix(mesh.points, connectivity)

print(f"{mesh.n_points} vertices, {mesh.n_cells} triangles")
print(f"{observed_vertex_indices.shape[0]} observed vertices")

15820 vertices, 31375 triangles
1000 observed vertices


## Build the posterior

Reference settings, matching `scripts/run_map.py`'s and `scripts/preprocess_data.py`'s defaults.

In [27]:
posterior_settings = PosteriorSettings(
    mesh=mesh,
    basis_vectors=basis_vectors,
    prior_settings=PriorSettings(
        mean_vector=prior_mean_angle_field,
        kappa=0.05,
        tau=10.0,
        seed=0,
    ),
    eikonal_settings=EikonalSolverSettings(
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1.0,
    ),
    likelihood_settings=LikelihoodSettings(
        num_vertices=mesh.points.shape[0],
        observed_vertex_indices=observed_vertex_indices,
        data_vector=observed_activation_times,
        noise_variance=1e-3,
    ),
)
posterior_builder = PosteriorBuilder(
    posterior_settings, interpolation_strategy=interpolation_strategy
)
log_posterior = posterior_builder.build()

## Build the sampler

`MCMCSettings.algorithm` selects the sampler (`MCMCAlgorithmName.PCN` or `MCMCAlgorithmName.MALA`); both use `step_width`, with a default far smaller than usual since the likelihood is very sharp relative to the broad prior (see `bayes_cep.mcmc.builder`). `tracked_components` adds the value and running mean of single mesh vertices to the logged outputs, next to the running acceptance rate and the mean over all components, which are always logged. `MCMCBuilder` writes the samples to `MCMC_DIR / storage.store_name` and overwrites an existing store by default.

In [28]:
mcmc_settings = MCMCSettings(
    algorithm=MCMCAlgorithmName.MALA,
    step_width=1e-7,
    tracked_components=(),
    storage=ZarrStorageSettings(chunk_size=100),
)
MCMC_DIR.mkdir(exist_ok=True)
mcmc_builder = MCMCBuilder(
    mcmc_settings, log_posterior, posterior_builder.prior, output_dir=MCMC_DIR
)

## Run the chain

> **Note:** every step needs a forward eikonal solve on the full mesh (MALA additionally an adjoint solve), so long chains take a long time. To only inspect the results of an earlier run, set `RUN_SAMPLER = False` in the next cell: it skips sampling and reads the existing Zarr store (the store is only created, and overwritten by default, when the sampler is built, i.e. when `RUN_SAMPLER` is `True`).

In [29]:
RUN_SAMPLER = False
NUM_SAMPLES = 1000
SEED = 0

if RUN_SAMPLER:
    with BaseLogger(LoggerSettings(), prefix="mcmc") as logger:
        sampler_settings = SamplerSettings(num_samples=NUM_SAMPLES, log_interval=1)
        sampler = mcmc_builder.build(logger=logger)
        sampler.run(
            initial_state=map_estimate,
            settings=sampler_settings,
            seed=SEED,
        )
    acceptance_rate_history = mcmc_builder.outputs[0].all_values
    mean_history = mcmc_builder.outputs[1].all_values
    print(f"final acceptance rate: {acceptance_rate_history[-1]:.3f}")

## Chain diagnostics

Running acceptance rate and the mean over all components of the state along the chain (only available if the chain was run in this session).

In [30]:
if RUN_SAMPLER:
    fig, (acceptance_axis, mean_axis) = plt.subplots(1, 2, figsize=(10, 4))
    acceptance_axis.plot(acceptance_rate_history)
    acceptance_axis.set_xlabel("iteration")
    acceptance_axis.set_ylabel("running acceptance rate")
    mean_axis.plot(mean_history)
    mean_axis.set_xlabel("iteration")
    mean_axis.set_ylabel("mean over all components [rad]")
    fig.tight_layout()

## Posterior mean and variance

Computed block-wise from the Zarr store, so the full sample array never has to fit in memory. The first `BURN_IN` samples are discarded. The fiber angle is $\pi$-periodic, so the axial mean and variance are used (`compute_axial_mean_and_variance_blockwise`): the mean is the direction of the mean resultant vector of the doubled angles, and the variance is $-\tfrac{1}{2}\ln R$ with $R$ its length.

In [31]:
BURN_IN = 100

samples = zarr.open_group(MCMC_DIR / mcmc_settings.storage.store_name, mode="r")["data"]
print(f"{samples.shape[0]} stored samples of dimension {samples.shape[1]}")

mcmc_mean, mcmc_variance = compute_axial_mean_and_variance_blockwise(
    samples, burn_in=BURN_IN, block_size=samples.chunks[0]
)
mcmc_std = np.sqrt(mcmc_variance)

1000 stored samples of dimension 15820


## Posterior mean vs. ground truth and MAP estimate

Compared on simplices (via `vertex_to_simplex_matrix`), as in `map.ipynb`: this is the representation the forward model actually sees.

In [32]:
ground_truth_angle_field_simplex = vertex_to_simplex_matrix @ ground_truth_angle_field
map_estimate_simplex = vertex_to_simplex_matrix @ map_estimate
mcmc_mean_simplex = vertex_to_simplex_matrix @ mcmc_mean

# All fields share one color range, spanning all of them.
angle_fields = (ground_truth_angle_field_simplex, map_estimate_simplex, mcmc_mean_simplex)
shared_clim = (
    min(field.min() for field in angle_fields),
    max(field.max() for field in angle_fields),
)

for name, field_values in (
    ("ground truth", ground_truth_angle_field_simplex),
    ("MAP estimate", map_estimate_simplex),
    ("MCMC mean", mcmc_mean_simplex),
):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["angle"] = field_values
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv", clim=shared_clim)
    plotter.add_text(name)
    plotter.show()

Widget(value='<iframe src="http://localhost:34355/index.html?ui=P_0x7f7bd819bb10_10&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:34355/index.html?ui=P_0x7f7b8005ccd0_11&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:34355/index.html?ui=P_0x7f7b8005e990_12&reconnect=auto" class="pyv…

## Axial error vs. ground truth

As in `map.ipynb`, the signed axial difference (`compute_axial_data_diff`) is used, since the fiber angle is $\pi$-periodic.

In [33]:
map_error = compute_axial_data_diff(map_estimate_simplex, ground_truth_angle_field_simplex)
mcmc_mean_error = compute_axial_data_diff(mcmc_mean_simplex, ground_truth_angle_field_simplex)
print(f"MAP estimate mean absolute error: {np.mean(np.abs(map_error)):.4f} rad")
print(f"MCMC mean mean absolute error:    {np.mean(np.abs(mcmc_mean_error)):.4f} rad")

plot_mesh = mesh.copy()
plot_mesh.cell_data["error"] = mcmc_mean_error
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="error", cmap="coolwarm", clim=(-np.pi / 2, np.pi / 2))
plotter.add_text("MCMC mean error")
plotter.show()

MAP estimate mean absolute error: 0.0400 rad
MCMC mean mean absolute error:    0.0401 rad


Widget(value='<iframe src="http://localhost:34355/index.html?ui=P_0x7f7b78b687d0_13&reconnect=auto" class="pyv…

## Posterior standard deviation

Square root of the axial variance, in radians.

In [35]:
plot_mesh = mesh.copy()
plot_mesh.point_data["std"] = mcmc_std
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="std", cmap="viridis")
plotter.add_text("MCMC pointwise axial standard deviation [rad]")
plotter.show()

Widget(value='<iframe src="http://localhost:34355/index.html?ui=P_0x7f7b41b891d0_15&reconnect=auto" class="pyv…

## Correlation length

Single correlation length of the posterior, from the samples after burn-in (see `bayes_cep.statistics.correlation_length`): the axial correlation of $\sin(2(\theta - \mu))$ is pooled over pairs of vertices, binned by graph distance along the mesh, and fitted with the Matérn ($\nu = 1$) correlation. The distance where the binned correlation first drops below $1/e$ is a model-free cross-check. `MAX_DISTANCE` should be a few times the expected correlation length (in mesh units), and `BOUNDARY_MARGIN` keeps base vertices away from the mesh boundary.

In [ ]:
MAX_DISTANCE = 150.0
BOUNDARY_MARGIN = 20.0

correlation = estimate_correlation_length(
    mesh.points,
    connectivity,
    samples,
    CorrelationLengthSettings(
        max_distance=MAX_DISTANCE,
        boundary_margin=BOUNDARY_MARGIN,
        burn_in=BURN_IN,
        block_size=samples.chunks[0],
    ),
)
print(f"fitted correlation length:   {correlation.length:.2f}")
print(f"correlation drops below 1/e: {correlation.crossing_length:.2f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(correlation.bin_distances, correlation.bin_correlations, "o", label="binned")
ax.plot(correlation.bin_distances, correlation.fitted_correlations, label="Matérn fit")
ax.set_xlabel("graph distance")
ax.set_ylabel("axial correlation")
ax.legend()
fig.tight_layout()